# 19. 재현 스크립트 검증 (14·16·17)

## 배경
- `scripts/train_blend.py`: 모델 학습 → 확률 섞기 → 제출 파일 저장을 한 번에 실행하는 재현 스크립트
- 프리셋 기준: **LB에 실제로 낸 파일 + 최종 후보만**
  - `blend14`: 14 XGB 0.7 + LR 0.3 → **LB 0.3247 (최종 후보)**
  - `select6`: 16 앙상블 선택(핫스팟 제외), 설아님 데스크탑 비율 RF 5 / LR_trunc 4 / XGB 3 / ET 2 (÷14) → LB 0.3183 (기록용, 최종 후보 아님)
  - `--xgb-seeds`: 17 XGB 시드 평균 (프리셋이 아닌 옵션, LB 미확인)

## 목적
- 스크립트가 노트북 14·16·17과 **같은 모델·비율·점수**를 만드는지 팀원 모두 확인할 수 있게 기록

## 검증 방법
- 이 노트북은 스크립트의 함수(`build_features`, `get_proba`, `PRESETS` 등)를 **그대로 불러와** 사용 → 스크립트와 따로 놀지 않음
- 제출 파일은 저장하지 않음 (점수·예측 비교만)
- 비교 기준
  - 모델별 CV → 노트북 14·16 출력 (설아님 데스크탑)
  - `blend14` OOF → 노트북 14 출력 / `select6` → 16 출력은 교차 적합 점수라 같은 기준 비교는 없음 (참고로만 표시)
  - 시드 평균 → 노트북 17 출력과 17 제출 파일
- 판단: LR 계열(무작위성 없음)은 완전히 같아야 하고, XGB가 들어간 값은 **실행 환경(CPU) 차이** 범위 안이면 재현으로 봄
  - 17 리뷰에서 확인: 같은 설정이라도 Windows(x86) XGB CV 0.4161 vs macOS M1(arm64) 0.4125
- 모델별 확률은 스크립트 캐시(`submissions/cache/`, 설정 해시 포함)를 사용 → 캐시가 없으면 새로 학습 (전체 약 20분)

#### 준비: 실행 환경 + 스크립트 불러오기
- 결과를 공유할 때 OS·CPU·버전을 같이 기록 (XGB 결과가 CPU에 따라 다름)

In [1]:
import sys; sys.path.append('..'); sys.path.append('../scripts')
import os, platform, numpy as np, pandas as pd, sklearn, xgboost as xgb
import train_blend as tb   # 재현 스크립트
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.validation import seed_everything, encode_target, get_folds, macro_f1

print(f'{platform.system()} {platform.release()} | CPU {platform.machine()} | python {platform.python_version()}')
print(f'xgboost {xgb.__version__} | scikit-learn {sklearn.__version__} | numpy {np.__version__} | pandas {pd.__version__}')
print('프리셋:', {k: {m: round(v, 3) for m, v in w.items()} for k, w in tb.PRESETS.items()})

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
folds = get_folds(y)
feats = tb.build_features(train, test, genes, {tb.MODELS[m][0] for m in tb.MODELS})
CACHE = tb.SUBMISSION_DIR / 'cache'
os.makedirs(CACHE, exist_ok=True)
print({k: v[0].shape for k, v in feats.items()})

Darwin 25.5.0 | CPU arm64 | python 3.12.14
xgboost 3.4.1 | scikit-learn 1.9.1 | numpy 2.5.4 | pandas 3.1.1
프리셋: {'blend14': {'XGB': 0.7, 'LR': 0.3}, 'select6': {'RF': 0.357, 'LR_trunc': 0.286, 'XGB': 0.214, 'ET': 0.143}}
{'any': (6201, 4384), 'any+trunc': (6201, 8768), 'any+stats': (6201, 4393)}


#### 1) 모델별 CV: 스크립트 vs 노트북 14·16 출력
- 노트북 출력(설아님 데스크탑, Windows)과 비교. 차이는 실행 환경 차이로 해석

In [2]:
REF = {'XGB': (0.4161, 0.0060, '14'), 'LR': (0.3818, 0.0064, '14'), 'LR_trunc': (0.3920, 0.0075, '16'),
       'RF': (0.3892, 0.0093, '14'), 'ET': (0.3793, 0.0152, '14')}   # 노트북 출력 (설아님 데스크탑)
P_oof, P_test, rows = {}, {}, []
for name in tb.MODELS:
    oof, tst, how = tb.get_proba(name, SEED, feats, y, folds, CACHE, use_cache=True)
    P_oof[name], P_test[name] = oof, tst
    cv, sd = tb.fold_cv(oof, y, folds)
    ref_cv, ref_sd, nb_no = REF[name]
    rows.append({'모델': name, '피처': tb.MODELS[name][0], '스크립트 CV': f'{cv:.4f} ± {sd:.4f}',
                 '노트북 CV': f'{ref_cv:.4f} ± {ref_sd:.4f} ({nb_no})', '차이': round(cv - ref_cv, 4), '확률 출처': how})
pd.DataFrame(rows)

,모델,피처,스크립트 CV,노트북 CV,차이,확률 출처
0,XGB,any+stats,0.4125 ± 0.0109,0.4161 ± 0.0060 (14),-0.0036,cache
1,LR,any,0.3818 ± 0.0065,0.3818 ± 0.0064 (14),-0.0000,cache
2,LR_trunc,any+trunc,0.3920 ± 0.0075,0.3920 ± 0.0075 (16),0.0000,cache
3,RF,any+stats,0.3905 ± 0.0100,0.3892 ± 0.0093 (14),0.0013,cache
4,ET,any+stats,0.3803 ± 0.0148,0.3793 ± 0.0152 (14),0.0010,cache


#### 2) 프리셋별 섞기 점수
- 비율은 노트북에서 정한 고정값 그대로 (다시 고르지 않음)
- `blend14`: 노트북 14 출력(전체 OOF)과 같은 기준으로 비교
- `select6`: 16 출력은 교차 적합 점수(비율 선택 포함)라 직접 비교 불가 → 참고로만 표시, 스크립트 OOF는 비율을 고른 OOF로 다시 매긴 값이라 낙관적

In [3]:
REF_BLEND = {'blend14': ('14 출력 OOF', 0.4231), 'select6': ('16 교차 적합 (참고)', 0.4261)}
LB = {'blend14': 0.3247, 'select6': 0.3183}
mix = lambda w, P: sum(v * P[m] for m, v in w.items())
rows = []
for preset, w in tb.PRESETS.items():
    oof = mix(w, P_oof)
    cv, sd = tb.fold_cv(oof, y, folds)
    o = macro_f1(y, oof.argmax(1))
    ref_name, ref = REF_BLEND[preset]
    rows.append({'프리셋': preset, '비율': {m: round(v, 3) for m, v in w.items()}, '스크립트 CV': f'{cv:.4f} ± {sd:.4f}',
                 '스크립트 OOF': round(o, 4), '노트북 기록': f'{ref:.4f} ({ref_name})',
                 '차이': round(o - ref, 4) if preset == 'blend14' else '-', 'LB': LB[preset]})
pd.DataFrame(rows)

,프리셋,비율,스크립트 CV,스크립트 OOF,노트북 기록,차이,LB
0,blend14,"{'XGB': 0.7, 'LR': 0.3}",0.4187 ± 0.0065,0.4190,0.4231 (14 출력 OOF),-0.0041,0.3247
1,select6,"{'RF': 0.357, 'LR_trunc': 0.286, 'XGB': 0.214, 'ET': 0.143}",0.4277 ± 0.0092,0.4274,0.4261 (16 교차 적합 (참고)),-,0.3183


#### 3) XGB 시드 5개 평균 (17): 스크립트 vs 노트북 17
- 17은 이 환경(M1)에서 실행 → 같은 환경이라 숫자와 예측이 완전히 같아야 함
- 17 제출 파일이 있으면 test 예측을 한 명씩 비교

In [4]:
SEEDS = [42, 0, 1, 2, 3]
xgb_oof, xgb_test = [], []
for s in SEEDS:
    oof, tst, how = tb.get_proba('XGB', s, feats, y, folds, CACHE, use_cache=True)
    xgb_oof.append(oof); xgb_test.append(tst)
    print(f'XGB seed {s}: CV {tb.fold_cv(oof, y, folds)[0]:.4f} ({how})')
P_seed = {'XGB': np.mean(xgb_oof, axis=0), 'LR': P_oof['LR']}
T_seed = {'XGB': np.mean(xgb_test, axis=0), 'LR': P_test['LR']}
w = tb.PRESETS['blend14']
cv, sd = tb.fold_cv(mix(w, P_seed), y, folds)
o = macro_f1(y, mix(w, P_seed).argmax(1))
print(f'\n시드 평균 섞기: 스크립트 CV {cv:.4f} ± {sd:.4f}, OOF {o:.4f} | 노트북 17: CV 0.4220 ± 0.0079, OOF 0.4223')

f17 = '../submissions/1111_blend_XGB5seed0.7_LR0.3_0.422.csv'
if os.path.exists(f17):
    pred = le.inverse_transform(mix(w, T_seed).argmax(1))
    same = (pd.read_csv(f17)['SUBCLASS'].to_numpy() == pred).mean()
    print(f'17 제출 파일과 test 예측 일치율: {same:.1%}')
else:
    print('17 제출 파일 없음 (다른 컴퓨터) → 숫자 비교만')

XGB seed 42: CV 0.4125 (cache)
XGB seed 0: CV 0.4126 (cache)
XGB seed 1: CV 0.4107 (cache)
XGB seed 2: CV 0.4145 (cache)
XGB seed 3: CV 0.4133 (cache)

시드 평균 섞기: 스크립트 CV 0.4220 ± 0.0079, OOF 0.4223 | 노트북 17: CV 0.4220 ± 0.0079, OOF 0.4223
17 제출 파일과 test 예측 일치율: 100.0%


#### 4) 캐시 설정 해시 확인
- 캐시 파일 이름 = `{모델}_s{시드}_{설정 해시}.npz`
- 해시에 들어가는 것: 모델 파라미터, 피처, 시드, `src/` 4개 파일 내용, 라이브러리 버전, CPU 종류
- 설정을 하나라도 바꾸면 해시가 달라져서 예전 캐시를 쓰지 않음 → 아래에서 파라미터 하나만 바꿔 확인

In [5]:
h_base = tb.config_hash('XGB', SEED)
orig = tb.XGB10['max_depth']
tb.XGB10['max_depth'] = 4                     # 설정 하나만 바꿔 봄
h_changed = tb.config_hash('XGB', SEED)
tb.XGB10['max_depth'] = orig                  # 원래대로
print(f'XGB 시드 42 해시: {h_base} | max_depth 3 → 4로 바꾸면: {h_changed} | 같은가: {h_base == h_changed}')
print(f'시드만 바꿔도 다른 해시: {tb.config_hash("XGB", 0)}')
print('원래 설정으로 되돌린 해시:', tb.config_hash('XGB', SEED), '(처음과 같음)')

XGB 시드 42 해시: 56714f90 | max_depth 3 → 4로 바꾸면: 0863fcc6 | 같은가: False
시드만 바꿔도 다른 해시: 917c068f
원래 설정으로 되돌린 해시: 56714f90 (처음과 같음)


## 결과

**1) 모델별 CV** (이 환경: macOS M1 arm64 / 노트북 출력: 설아님 Windows 데스크탑)

| 모델 | 스크립트 CV | 노트북 CV | 차이 |
| :--- | :-: | :-: | :-: |
| XGB (any + stats) | 0.4125 ± 0.0109 | 0.4161 ± 0.0060 (14) | −0.0036 |
| LR (any) | 0.3818 ± 0.0065 | 0.3818 ± 0.0064 (14) | 0.0000 |
| LR_trunc (any + trunc) | 0.3920 ± 0.0075 | 0.3920 ± 0.0075 (16) | 0.0000 |
| RF (any + stats) | 0.3905 ± 0.0100 | 0.3892 ± 0.0093 (14) | +0.0013 |
| ET (any + stats) | 0.3803 ± 0.0148 | 0.3793 ± 0.0152 (14) | +0.0010 |

- LR 계열(무작위성 없음)은 노트북과 완전히 같음 → 데이터·피처·fold가 같다는 확인
- XGB −0.0036은 17 리뷰에서 확인한 실행 환경 차이와 같은 크기, RF·ET도 ±0.001 수준의 환경 차이

**2) 프리셋별 섞기**

| 프리셋 | 비율 | 스크립트 OOF | 노트북 기록 | LB |
| :--- | :--- | :-: | :-: | :-: |
| blend14 (최종 후보) | XGB 0.7 + LR 0.3 | 0.4190 | 0.4231 (14 출력 OOF, 차이 −0.0041) | 0.3247 |
| select6 (기록용) | RF 5 / LR_trunc 4 / XGB 3 / ET 2 (÷14) | 0.4274 | 0.4261 (16 교차 적합, 참고) | 0.3183 |

- blend14 차이는 XGB 환경 차이에서 나옴 (XGB 비중 0.7)
- select6 스크립트 OOF는 비율을 고른 OOF로 다시 매긴 값이라 낙관적 → 기록에는 16 교차 적합 0.4261 사용
- select6은 OOF가 더 높지만 LB는 blend14보다 −0.006 → "CV는 오르는데 LB는 안 오르는" 패턴 재확인

**3) XGB 시드 5개 평균 (17, 옵션)**: 스크립트 CV 0.4220 ± 0.0079, OOF 0.4223 → 노트북 17과 **완전히 같음**, 17 제출 파일과 test 예측 **100% 일치** (같은 환경)

**4) 캐시 설정 해시**: `max_depth`만 3 → 4로 바꿔도 해시가 달라지고(56714f90 → 0863fcc6), 시드만 바꿔도 다름 → 설정이 바뀌면 예전 캐시를 쓰지 않음

## 결론
- **재현 스크립트는 노트북 14·16·17과 같은 모델·비율을 만듦**
  - 같은 환경(17)에서는 점수와 test 예측이 완전히 일치
  - 다른 환경(설아님 데스크탑)과의 차이는 XGB 실행 환경 차이만큼이고, LR 계열은 완전히 같음
- 팀 규칙: 비율은 어느 컴퓨터에서 골랐는지 적고 고정, **제출 파일은 설아님 Windows 데스크탑에서 `--no-cache`로 생성** → 그 환경에서 이 노트북을 실행하면 노트북 출력과 같은 숫자가 나와야 함 (확인 요청)
- 이 노트북의 확률은 스크립트 캐시(이 컴퓨터에서 스크립트 함수로 학습한 결과)를 사용 → 다른 컴퓨터에서는 캐시가 없어 새로 학습 (약 20분)
